# 04 — Auditoría Lógica, Deducción y Reconciliación Matemática (Agente 3)

**Objetivo:** Verificar las reglas de negocio reglamentarias (FATARCO / World Archery) implementadas en el **Agente 3**:
1. **Deducción de flechas ilegibles:** reconstrucción exacta cuando 1 de 3 flechas tiene ruido OCR pero el parcial escrito es válido (ej. `2 + 3 + 'U' = 9` $\rightarrow$ `'4'`).
2. **Reconciliación OCR `5` vs. `6`:** corrección automática ante ambigüedades visuales habituales.
3. **Deducción del Total General y Conteo `X`/`10`/`9`:** diferenciación entre modalidad **Sala (Indoor)** y **Aire Libre (Outdoor)**.
4. **Cálculo del Índice de Certeza (`0%`–`100%`)** para enrutamiento automático vs. revisión humana.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd

try:
    from tiro_arco.agente_3_validador_conciliador import Agente3ValidadorConciliador
except ImportError:
    from agente_3_validador_conciliador_2 import Agente3ValidadorConciliador

agente3 = Agente3ValidadorConciliador(padron_arqueros={
    '14A': {'arquero': 'Monserrat Ana Gabriela', 'club': 'AFPSF', 'categoria': 'B E80'}
})

manifiesto_2_simulado = {
    'archivo_origen': 'test_reglas_agente3.pdf',
    'transcripciones': [
        {
            'pagina': 1,
            'target_id': '14A',
            'transcripcion': {
                'arquero': 'Desconocido',
                'target': '14A',
                'club': 'N/A',
                'categoria': 'N/A',
                'distancia': '18m',
                'ronda': '1',
                'series': [
                    {'serie': 1, 'flechas': ['2', '3', 'U'], 'suma_parcial_escrita': 9},
                    {'serie': 2, 'flechas': ['6', '5', '5'], 'suma_parcial_escrita': 15},
                    {'serie': 3, 'flechas': ['X', '10', '9'], 'suma_parcial_escrita': 29, 'total_acumulado_escrito': 53}
                ],
                'totales_planilla': {'total_puntos': 0, 'cant_10s': 0, 'cant_9s': 0}
            }
        },
        {
            'pagina': 1,
            'target_id': '14B',
            'transcripcion': {
                'arquero': 'Arquero Aire Libre',
                'target': '14B',
                'club': 'AFAPIECO',
                'categoria': 'R M',
                'distancia': '70m',
                'ronda': '1',
                'series': [
                    {'serie': 1, 'flechas': ['X', 'X', '10'], 'suma_parcial_escrita': 30, 'total_acumulado_escrito': 30}
                ],
                'totales_planilla': {'total_puntos': 30, 'cant_10s': 0, 'cant_9s': 0, 'cant_xs': 0}
            }
        }
    ]
}

manifiesto_3 = agente3.auditar_manifiesto_completo(manifiesto_2_simulado)
print('Resumen Auditoría:', json.dumps(manifiesto_3['resumen_auditoria'], indent=2, ensure_ascii=False))

for item in manifiesto_3['resultados_auditados']:
    print(f"\n=== Diana {item['target_id']} | Arquero: {item['arquero']} | Modalidad: {item['modalidad']} ===")
    print(f"Score Certeza: {item['score_certeza']}% | Estado: {item['estado_auditoria']} | Matcheado Padrón: {item['matcheado_por_padron']}")
    print('Totales Calculados:', item['totales_calculados_series'])
    for adv in item['advertencias']:
        print('  -', adv)